# 07 — Real shift: training and test records from different patients (Parkinson's)
Notebooks 02–06 create the shift artificially (Gan & Liu's tilting). Here the shift comes from the
data itself: the model is trained on one group of patients and tested on another group.

| Design | Training patients | Test patients | What it tests |
|---|---|---|---|
| `random` | random half of the 42 patients | the other half | only "new patients", no systematic shift (reference) |
| `age` | younger than the median patient age | the older patients | a real covariate shift |
| `sex` | sex = 0 | sex = 1 | a real covariate shift |

**[my choice]** these three designs; 1000 training and 1000 test records per split (fewer if the
group is smaller), 20 splits. **[my reading]** UCI codes sex as 0 = male, 1 = female.

**Important caveat.** Covariate shift assumes the same Y | X in both groups. With different
patients this may not hold exactly (patient effects), and records of the same patient are
correlated. So this notebook tests the methods outside their assumptions — that is the point of a
"real" benchmark, and the `random` design shows how much of any coverage loss comes from new
patients alone.

Age and sex are NOT inputs of the model (inputs = the 16 voice features), so the density ratio is
estimated from the voice features only.

**Patient-grouped calibration (`GROUPED_CAL = True`).** The first full run (record-level splits inside
the training data) under-covered for every method, even in the `random` design: the calibration
records came from the same patients as the training records, so the calibration scores did not
include the error of predicting a NEW patient. With `GROUPED_CAL = True` every internal split keeps
each patient on one side **[my choice]**: network training / early-stopping / calibration patients are
different; SCP fit and calibration patients are different; EU uses leave-one-patient-out; EUc-LR uses
a patient (cluster) bootstrap with an out-of-bag pseudo-test record; OTCP/GRCP hold-out and
calibration patients are different. Results go to a separate folder, so the first run is kept.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

sys.path.insert(0, os.path.abspath("../src"))
import cpshift as cp
import experiment as ex
import deep

print("device:", deep.DEVICE)

## Configuration (network and method settings as in notebook 04)

In [ ]:
QUICK = True
GROUPED_CAL = True     # False = the first run (record-level internal splits)

CFG = dict(
    SEED=2026, ALPHA=0.10, N_TRAIN=1000, N_TEST=1000, N_REPS=20, B=1000, SCP_FIT_FRAC=0.5,
    DEEP_CAL_FRAC=0.5, DEEP_VAL_FRAC=0.2, HIDDEN=64, N_LAYERS=2, LR=1e-3, WEIGHT_DECAY=1e-4,
    BATCH=64, MAX_EPOCHS=500, PATIENCE=30, EPS=1e-3,
    OT_M=1024, OT_EPS=0.1, GRID_SIDE=150, GRCP_VOL_POINTS=100,
)
DESIGNS = ['random', 'age', 'sex']
CC = dict(WSC_DELTA=0.10, WSC_N_DIR=1000, WSC_GRID=50, CEC_J=10)
if QUICK:
    CFG.update(N_REPS=1, B=200, MAX_EPOCHS=40, GRCP_VOL_POINTS=20)
    CC.update(WSC_N_DIR=100)

OUT_DIR = "../results/patient_shift_grouped" if GROUPED_CAL else "../results/patient_shift"
os.makedirs(OUT_DIR, exist_ok=True)
CFG

## The patient groups

In [ ]:
df, X_all, Y_all = cp.load_parkinsons("../data/parkinsons_updrs.data")
pat = df.groupby('subject#')[['age', 'sex']].first()
print("patients:", len(pat), "| records:", len(df))
print("median patient age:", np.median(pat['age']))
print("patients by sex:", pat['sex'].value_counts().to_dict())
for design in ['age', 'sex']:
    src, tgt = ex.patient_groups(df, design, np.random.default_rng(0))
    ns = df['subject#'].isin(src).sum(); nt = df['subject#'].isin(tgt).sum()
    print(f"{design}: {len(src)} source patients ({ns} records) -> {len(tgt)} target patients ({nt} records)")

## Run

In [ ]:
res_df = ex.run_patient_shift(df, X_all, Y_all, DESIGNS, CFG, CC, deep, grouped_cal=GROUPED_CAL)
res_df.to_csv(f"{OUT_DIR}/results_all_reps.csv", index=False)

## Tables

In [ ]:
summ = (res_df.groupby(['design', 'method'])
        .agg(coverage=('coverage', 'mean'), coverage_sd=('coverage', 'std'),
             volume=('volume_median', 'median'), volume_mean=('volume_median', 'mean'),
             pct_infinite=('pct_infinite', 'mean'), WSC=('WSC', 'mean'), CEC_X=('CEC_X', 'mean'),
             ess_cal=('ess_cal', 'mean'), n_cal=('n_cal', 'mean'))
        .round(2).reset_index())
summ.to_csv(f"{OUT_DIR}/summary.csv", index=False)
table = summ.pivot(index='method', columns='design', values=['coverage', 'volume', 'WSC']).reindex(ex.METHODS_ALL)
table = table.reorder_levels([1, 0], axis=1)[[(d, c) for d in DESIGNS for c in ['coverage', 'volume', 'WSC']]]
table.to_csv(f"{OUT_DIR}/table_main.csv")
print(table.to_string())
print("\nESS of the calibration weights (mean over splits):",
      summ.groupby('design').ess_cal.first().to_dict(), "out of", int(summ.n_cal.iloc[0]))

## Paired comparisons (ours vs the best shift-corrected baselines), per design

In [ ]:
pairs = [('Deep-LocalCov + WCP (ours)', 'EUc-LR'), ('Deep-LocalCov + EUc (ours)', 'EUc-LR'),
         ('Deep-LocalCov + WCP (ours)', 'OTCP + WCP'), ('Deep-LocalCov + EUc (ours)', 'OTCP + WCP'),
         ('Deep-LocalCov + WCP (ours)', 'Deep-LocalCov'), ('Deep-LocalCov + EUc (ours)', 'Deep-LocalCov')]
rows = []
for design in DESIGNS:
    d = res_df[res_df.design == design].pivot(index='rep', columns='method',
                                                values=['coverage', 'volume_median', 'WSC'])
    for m1, m2 in pairs:
        for metric in ['coverage', 'volume_median', 'WSC']:
            x = (d[metric][m1] - d[metric][m2]).to_numpy()
            x = x[np.isfinite(x)]
            p = stats.ttest_1samp(x, 0).pvalue if len(x) > 1 else np.nan
            rows.append(dict(design=design, comparison=f"{m1} - {m2}", metric=metric,
                             mean_diff=x.mean(), n=len(x), p_t=p))
paired = pd.DataFrame(rows).round(4)
paired.to_csv(f"{OUT_DIR}/paired.csv", index=False)
print(paired.to_string())

## Figure: coverage and area per design

In [ ]:
show = ['EU', 'EUc-LR', 'OTCP', 'OTCP + WCP', 'GRCP + WCP', 'Deep-LocalCov',
        'Deep-LocalCov + WCP (ours)', 'Deep-LocalCov + EUc (ours)']
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
xpos = np.arange(len(DESIGNS)); wbar = 0.8 / len(show)
for j, m in enumerate(show):
    s = summ[summ.method == m].set_index('design').reindex(DESIGNS)
    ax[0].bar(xpos + j * wbar, s.coverage, wbar, yerr=s.coverage_sd, label=m)
    ax[1].bar(xpos + j * wbar, s.volume, wbar)
ax[0].axhline(90, color='k', lw=1, ls=':'); ax[0].set_ylim(75, 100)
ax[0].set_ylabel('coverage (%)'); ax[1].set_ylabel('median area (UPDRS$^2$)'); ax[1].set_yscale('log')
for x in ax:
    x.set_xticks(xpos + 0.4 - wbar / 2); x.set_xticklabels(DESIGNS)
fig.legend(loc='lower center', ncol=4, fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.12))
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig_patient_shift.png", dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
print("Files saved in", os.path.abspath(OUT_DIR), ":", sorted(os.listdir(OUT_DIR)))
print("GROUPED_CAL:", GROUPED_CAL)
print("CONFIG:", CFG)